In [ ]:
import pickle
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

import easyMPRA_20231120 as em
from _helper_functions import *

In [ ]:
dnaMinRpmThresh=0.05 # do not consider any barcodes with RPM <.05
minDnaBc=5 # do not consider any enhancers with less than 5 barcodes per enhancer
minNumRepsDetectedRna=3 # to consider an enhancer in the analysis, require it is detected in at least 3 replicates
ratioAggFuncTuple2minActivityToCount={'mean':1.5} # require the enhancer has a minimum of 1.5 activity. anything below this is noise


In [ ]:
endf=pd.read_pickle(f'01-Barcode-Processing/endf.pd.df.pickle')

In [ ]:
len(endf)

In [ ]:
endf.head(1)

In [ ]:
# this dictionary allows us to access the different samples in for loops
SampleScaffoldKey2ValueList={
    '{nt}':['R','D'],
    '{rep}':['1','2','3','4','5','6','7'],
}


In [ ]:


scaffold='rpm-{nt}{rep}'

for rep in SampleScaffoldKey2ValueList['{rep}']:
    
    print(rep)
    
    # without dna rpm min rilter
    dna=scaffold.replace('{nt}','D').replace('{rep}',rep)
    rna=scaffold.replace('{nt}','R').replace('{rep}',rep)
    
    endf[rna+'-filtNa-Dnotnull'],\
    endf[dna+'-filtNa-Dnotnull']=\
    em.filter_na_matched(indf=endf,
                         col1=rna,
                         col2=dna,
                         treatNaType={rna:'zero',dna:'omit'})
    
    # dna rpm min rilter

    endf[dna+f'-minDnaRpm{dnaMinRpmThresh}']=em.mask_na_filter(endf,dna,filterType='min',minVal=dnaMinRpmThresh)
    
    endf[rna+f'-minDnaRpm{dnaMinRpmThresh}'+'-filtNa-Dnotnull'],\
    endf[dna+f'-minDnaRpm{dnaMinRpmThresh}'+'-filtNa-Dnotnull']=\
    em.filter_na_matched(indf=endf,
                         col1=rna,
                         col2=dna+f'-minDnaRpm{dnaMinRpmThresh}',
                         treatNaType={rna:'zero',dna+f'-minDnaRpm{dnaMinRpmThresh}':'omit'})


# filter out any enhancers with <5 dna barcodes

In [ ]:
scaffold = f'rpm-{{nt}}{{rep}}-minDnaRpm{dnaMinRpmThresh}-filtNa-Dnotnull'
    
for rep in SampleScaffoldKey2ValueList['{rep}']:

    print(rep)

    thisColDNA=scaffold.replace('{nt}','D').replace('{rep}',rep)
    thisColRNA=scaffold.replace('{nt}','R').replace('{rep}',rep)

    thisColDNAOut=thisColDNA+f'-minDnaBc{minDnaBc}'
    thisColRNAOut=thisColRNA+f'-minDnaBc{minDnaBc}'

    endf[thisColDNAOut]=endf.apply(lambda row: row[thisColDNA] if len(row[thisColDNA])>=minDnaBc else [],axis=1)
    endf[thisColRNAOut]=endf.apply(lambda row: row[thisColRNA] if len(row[thisColDNA])>=minDnaBc else [],axis=1)




In [ ]:
def _opwrapper(l,func,omitna=True,null=np.NaN):
    # remove na if desired
    if omitna:    l=[li for li in l if not pd.isnull(li)]
    
    # 
    if len(l)>=1: return func(l)
    else:         return null

def _sum(l):
    return _opwrapper(l,func=sum,null=0)

def _max(l):
    return _opwrapper(l,func=max,null=0)

def _mean(l):
    return _opwrapper(l,func=np.mean,null=0)

In [ ]:

# ratio 

aggfunc,agglabel=_mean,'mean' # aggregate within each rep by taking the mean of all RNA:DNA ratios, ignoring NA values

scaffold=f'rpm-{{nt}}{{rep}}' 
filtstyle=f'-minDnaRpm{dnaMinRpmThresh}-filtNa-Dnotnull-minDnaBc{minDnaBc}'
allRepsString='1234567'
        
# calculat reps individually
for rep in SampleScaffoldKey2ValueList['{rep}']:

    rna = scaffold.replace('{nt}','R').replace('{rep}',rep) + filtstyle
    dna = scaffold.replace('{nt}','D').replace('{rep}',rep) + filtstyle

    listcol=f'ratio-replist-{rep}{filtstyle}'

    endf[listcol] = \
    em.aggregate_matched_across_samples(indf=endf, 
                                        col1=rna, 
                                        col2=dna, 
                                        aggfunc=em.ratio)

    aggcol=f'ratio-repagg-{rep}{filtstyle}-{agglabel}'
    endf[aggcol] = \
    em.aggregate_within_samples(indf=endf,col=listcol,aggfunc=aggfunc)

# aggregate reps
minActivityToCount=ratioAggFuncTuple2minActivityToCount[agglabel]
expWideScaffold=f'ratio-repagg-{{rep}}{filtstyle}-{agglabel}'
colsToCombine=em.label_generator(expWideScaffold,SampleScaffoldKey2ValueList,'{rep}',endf.columns)

# takes sum of acitivity-wide replicate to get an expirement-wide activity score
allRepsString=''.join(SampleScaffoldKey2ValueList['{rep}'])
endf[f'ratio-expagg-{allRepsString}{filtstyle}-{agglabel}'] = em.concat_sample_values(endf,colsToCombine)
endf[f'ratio-expagg-{allRepsString}{filtstyle}-{agglabel}-minActFilt{minActivityToCount}'] = em.mask_na_filter(endf,f'ratio-expagg-{allRepsString}{filtstyle}-{agglabel}',minVal=minActivityToCount,filterType='min')
endf[f'ratio-expagg-{allRepsString}{filtstyle}-{agglabel}-minActFilt{minActivityToCount}-minRepsActObs{minNumRepsDetectedRna}-expsum'] = endf[f'ratio-expagg-{allRepsString}{filtstyle}-{agglabel}-minActFilt{minActivityToCount}'].apply(lambda l: sum    ([li for li in l if pd.notnull(li)]) if len([li for li in l if pd.notnull(li)])>=minNumRepsDetectedRna else np.NaN)

In [ ]:
endf.head(1)

# rescale activity so <0 is inert and >1 is active

In [ ]:

minRepsActObs=3
dnaMinRpmThresh=.05
minSumRnaSumDnaFilt=1.5
minDnaBc=5
pctlmin,pctlmax=(55,65)
aggfunc1='mean'
aggfunc2='sum'

activityColFinal=f'ratio-expagg-1234567-minDnaRpm{dnaMinRpmThresh}-filtNa-Dnotnull-minDnaBc{minDnaBc}-{aggfunc1}-minActFilt{minSumRnaSumDnaFilt}-minRepsActObs{minRepsActObs}-exp{aggfunc2}'     
endf[activityColFinal+'-log2']=endf[activityColFinal].apply(lambda a: np.log2(a) if a!=0 else np.NaN)


In [ ]:
Func2Color={'Neural + Ectopic': cb['pink'],
 'Neural': cb['green'],
 'Inert': cb['orangenature'],
}

skipControls=['Not Tested','Weak Neural','Inert - No Counting','Nueral - No Counting']

In [ ]:
# rescale based on dual cutoffs
activityColName=activityColFinal+'-log2'

activityName=activityColName

plotTitle=''

thresholdToLabelPositives = np.percentile(endf[activityColName].dropna(), pctlmax)
thresholdToLabelNegatives = np.percentile(endf[activityColName].dropna(), pctlmin)

log2transformactivity=False
ymin,ymax=-5,3

printBadControls=False
plot_legend=False

subsampleviolin=False
scaling_range = thresholdToLabelPositives - thresholdToLabelNegatives
endf[activityColName+f'-rescaled-minpctl{pctlmin}-maxpctl{pctlmax}'] = endf[activityColName].apply(lambda a:(a - thresholdToLabelNegatives) / scaling_range)



In [ ]:
# show repro of rescaling

# show that the scaling is correct
fig,ax=plt.subplots(1)

ax.scatter(endf[activityColName],endf[activityColName+f'-rescaled-minpctl{pctlmin}-maxpctl{pctlmax}'])
ax.axvline(thresholdToLabelNegatives,color='red')
ax.axvline(thresholdToLabelPositives,color='blue')
ax.axhline(0,color='red')
ax.axhline(1,color='blue')

In [ ]:
activityColName=activityColName+f'-rescaled-minpctl{pctlmin}-maxpctl{pctlmax}'


In [ ]:
outcols=['enhancer-seq','enhancer-id','barcode-seq','ratio-expagg-1234567-minDnaRpm0.05-filtNa-Dnotnull-minDnaBc5-mean-minActFilt1.5-minRepsActObs3-expsum-log2-rescaled-minpctl55-maxpctl65']


fn=f'01-Barcode-Processing/endf_final-experiment-enhancer-activity'
endf.to_pickle(fn+'.pd.df.pickle')
endf.to_csv(fn+'.tsv',sep='\t',index=None)

# Plotting activity of known controls

In [ ]:
fn='../../../otxa-scrambled-constant/0-redo-analysis-for-nature-submission/1-rna-dna-bulk/data/all-control-names-and-sequences.Ols100Enhancer2Name.pydict.pickle'
with open(fn,'rb') as f: Ols100Enhancer2Name=pickle.load(f)

fn='../../../otxa-scrambled-constant/0-redo-analysis-for-nature-submission/1-rna-dna-bulk/data/20230926-Moderate-controls-ols-100_MicroscopeSeq2Func.pydict.pickle'
with open(fn,'rb') as f: Ols100Enhancer2FxnalGroup=pickle.load(f)

dprint(Ols100Enhancer2Name)
dprint(Ols100Enhancer2FxnalGroup)


In [ ]:
fn='ref/MicroscopeGroup2Seq.pydict.pickle'
MicroscopeGroup2Seq=load_pickle_dict(fn)

fn='ref/MicroscopeSeq2Func.pydict.pickle'
MicroscopeSeq2Func=load_pickle_dict(fn)

fn='ref/MicroscopeSeq2Name.pydict.pickle'
MicroscopeSeq2Name=load_pickle_dict(fn)
MicroscopeName2Seq={v:k for k,v in MicroscopeSeq2Name.items()}

In [ ]:
fn='/tscc/nfs/home/solvason/projects/otxa/otxa-scrambled-constant/0-redo-analysis-for-nature-submission/1-rna-dna-bulk/data/20230925-all-control-names-and-sequences.MicroscopeName2Seq.pydict.pickle'
olsMicroscopeName2Seq=load_pickle_dict(fn)


In [ ]:
# add ref data to dataframe

ctrlfxn=[]
ctrlname=[]

for seq in endf['enhancer-seq']:
    if seq in Ols100Enhancer2FxnalGroup:   ctrlfxn.append(Ols100Enhancer2FxnalGroup[seq])
    else:                                  ctrlfxn.append(np.NaN)
        
    if seq in Ols100Enhancer2Name:         ctrlname.append(Ols100Enhancer2Name[seq])
    else:                                  ctrlname.append(np.NaN)

endf['CONTROL_FUNCTION_INITIAL']=ctrlfxn
endf['CONTROL_FUNCTION']=ctrlfxn
endf['CONTROL_NAME']=ctrlname

In [ ]:
actlist=[c for c in endf.columns if 'ratio' in c and '1234' not in c and (('repagg' in c) or ('exp' in c))]
actlist

In [ ]:

####################################################################################
# Creaete DFs
####################################################################################

ctrlDF=endf.loc[endf.CONTROL_FUNCTION.notnull(),['CONTROL_FUNCTION','CONTROL_NAME',activityColName]]


ctrlDF=ctrlDF.replace(0,np.NaN)

# Subset only controls with good microscope data
if printBadControls==False:
    badControlSlice=ctrlDF.CONTROL_FUNCTION.str.contains('\*')
    unplottedGroups=ctrlDF.CONTROL_FUNCTION.isin(skipControls)
    print(sum(badControlSlice),'controls removed')
    print(' ',', '.join(ctrlDF.loc[badControlSlice,:].index.tolist()))
    ctrlDF=ctrlDF.loc[~badControlSlice & ~unplottedGroups,:]
    
print(len(ctrlDF),'controls used')
    
print(len(ctrlDF),'controls plotted...\n')
print(ctrlDF.CONTROL_NAME.tolist())

endf=endf.set_index('CONTROL_NAME',drop=False)


####################################################################################
# Plot all activities
####################################################################################

activity=activityColName

fig,ax=plt.subplots(1,figsize=(4,4),dpi=150)

#################################
# plot all activity
#################################

print('\tprepping all data...\n')
violinpos=-.3

if not subsampleviolin:
    data=endf[activity].dropna().tolist()
else:
    data=endf[activity].sample(subsampleviolin).dropna().tolist()


rmvalues=set(['inf','-inf',0])
data=[di for di in data if str(di) not in rmvalues]

print('\tplotting all data...\n')
parts=ax.violinplot(data,positions=[violinpos],widths=[.25],showmeans=False, showmedians=False, showextrema=False)
ax.scatter(violinpos,endf.at['OLS WT',activity],s=15,color=cb['green'])
# ax.scatter(violinpos,endf.at['OLS WT_ablatedGataEts',activity],s=15,color=cb['red'])

for pc in parts['bodies']:
    pc.set_facecolor('lightgrey')
    pc.set_edgecolor('grey')
    pc.set_alpha(1)

ax.set_ylabel(activityName)
ax.set_xlabel('')

ax.spines.right.set_visible(False)
ax.spines.top.set_visible(False)


#################################
# plot controls
#################################

print('\tprepping/plotting control data...\n')

# Create DF for swarmplot (groups on different X)
c2v={'controlname':ctrlDF.CONTROL_NAME,'function':ctrlDF.CONTROL_FUNCTION,'X':[0 for func in ctrlDF.CONTROL_FUNCTION],'Y':ctrlDF[activityColName]}

swarmDF=pd.DataFrame(c2v).dropna()

# print(swarmDF.dropna())

for x,y,func in zipdf(swarmDF,['X','Y','function']):

    color=Func2Color[func]
    ax.scatter(x,y,color=color)

# ax.set_ylim(-12,5)

# sns.swarmplot(data=swarmDF,x='X',y='Y',hue='function',ax=ax,palette=Func2Color,size=6)
# sns.swarmplot(data=swarmDF,x='X',y='Y',ax=ax,size=6)
# ax.spines.right.set_visible(False)
# ax.spines.top.set_visible(False)

# print(ctrlDF.loc[:,['CONTROL_NAME','CONTROL_FUNCTION',activityColName]].sort_values(activityColName).reset_index(drop=True))


# ####################################################################################
# # Legend
# ####################################################################################

# if plot_legend:
#     handles,labels = ax.get_legend_handles_labels()
#     order=['Neural + Ectopic Expression','Neural Enhancer','Weak Neural Enhancer','Non-Functional']
#     order=[labels.index(fxn) for fxn in order if fxn in labels]
#     plt.legend([handles[i] for i in order],[labels[i] for i in order],bbox_to_anchor=(1,1))
# else: ax.get_legend().remove()

# ####################################################################################
# # Plot thresholds for classification
# ####################################################################################

if thresholdToLabelPositives!=None: 
    ax.axhline(1,color=cb['green'],ls='--',lw=1.6)
if thresholdToLabelNegatives!=None: 
    ax.axhline(0,color=cb['red'],ls='--',lw=1.6)


numTotEns=len(endf)
numNullEns=sum(endf[activityColName].isnull())
percenNullEns=percent(numNullEns/len(endf))

numEnsWithActivity=len(endf)-numNullEns

nulEns=endf[activityColName].isnull()
confidentActList=endf.loc[~nulEns,activityColName].tolist()

numFunc=len([i for i in confidentActList if i>0])
numInert=len([i for i in confidentActList if i<=0])

f'{numInert:,} Inert',f'{numFunc:,} Active ens'

activityColTitle=activityColName.replace(aggfunc1,aggfunc1+'\n')


# ax.set_title(title)

# ax.set_ylabel('activity',size=7)

ax.set_xlabel('')
ax.set_ylabel('')
ax.set_xticks([])
ax.set_yticks([-10,-5,0,1,5,10,15,20,25])

ax.set_xticklabels([])
ax.set_yticklabels([])

# out='000-science-svgs/v2-2cutoffs_02_PRO_20231120_Enhancer-Activity-Analysis__genomic-activity_2cutoffs.svg'
# plt.savefig(out,format='svg',transparent=True)

plt.show()

In [ ]:
# controlsDetected=['OLS 41 ABL','OLS 85','OG-RC','OLS 12','OLS 19','OLS 11','OLS 22','OLS 29']

In [ ]:
# swarmDF

In [ ]:
# print(activityColName)
# ctrlDF.sort_values(['CONTROL_FUNCTION',activityColName]).loc[:,['CONTROL_NAME','CONTROL_FUNCTION',activityColName]]


In [ ]:
# fn=f'01b-generated-tables/{datestring}-ctrldf-filtered__activity={activityColName}.tsv'
# ctrlDF.sort_values(['CONTROL_FUNCTION',activityColName]).loc[:,['CONTROL_NAME','CONTROL_FUNCTION',activityColName]].to_csv(fn,index=True,sep='\t')


In [ ]:
# activityColName

In [ ]:
# fn=f'v2-2cutoffs_02-activities/{datestring}-endf-filtered__activity={activityColName}'
# endf.loc[:,['enhancer-seq','enhancer-id','CONTROL_NAME','CONTROL_FUNCTION',activityColName]].to_pickle(fn+'.pd.df.pickle')
# endf.loc[:,['enhancer-seq','enhancer-id','CONTROL_NAME','CONTROL_FUNCTION',activityColName]].to_csv(fn+'.tsv',index=False,sep='\t')

In [ ]:
# endf.loc['num-tot-olap'].value_counts()

In [ ]:
# endf.head()

In [ ]:
# endf.head(20)